In [11]:
# =============================================================================
# CELL 0: Imports
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
import warnings

In [12]:
# =============================================================================
# CELL 1: Configuration and Setup (Economic Indicators Data)
# =============================================================================

NOTEBOOK_NAME = "06: Economic Indicators Data Ingestion"
DATA_SUBFOLDER = "economy"
OBJECTIVE = """Process Statistik Austria economic indicators (monthly)
18 economic variables including production indices, trade, employment
Date range: 2009-2025

IMPORTANT ARCHITECTURAL DECISION:
Economic data is MONTHLY ONLY - no daily or weekly rows will be created.
Consistent with climate, production, and gas data architecture.
Economic variables can only be used in monthly-level analyses.

DATE FORMAT: mmm.yy (e.g., Jän.09) with Austrian German month abbreviations
COMPLEX STRUCTURE: Multiple header rows and footer metadata require careful skipping."""

# Economic data specific configuration
ECON_FILE_NAME = "table_2025-09-16_13-30-40.xlsx"
DATE_COLUMN = "Berichtszeitraum"
SKIPROWS = list(range(9)) + [10, 11]  # Skip rows 0-8 (metadata), 10-11 (wertangabe)
NROWS = 213  # Stop at row 213 (metadata text follows)
USECOLS = list(range(1, 20))  # Skip column 0 (empty), use columns 1-19

# Austrian German month abbreviations mapping (extended from gas prices)
GERMAN_MONTHS = {
    'Jän': '01', 'Jan': '01', 'Feb': '02', 'Mär': '03', 'Mrz': '03',
    'Apr': '04', 'Mai': '05', 'Jun': '06', 'Jul': '07',
    'Aug': '08', 'Sep': '09', 'Okt': '10', 'Nov': '11', 'Dez': '12'
}

# Column mapping: Source column name → Target column name
COLUMN_MAPPING = {
    'Berichtszeitraum': 'date',
    'Produktionsindex Industrie (at; 2021=100)': 'econ_prod_index_industry',
    'Verbraucherpreisindex (2015=100)': 'econ_consumer_price_index',
    'Ausfuhren Insgesamt in €': 'econ_exports_total_EUR',
    'Umsatzindex Handel (nom.; 2021=100)': 'econ_turnover_index_commercial_sales',
    'Nächtigungen': 'econ_count_overnight_stays',
    'Umsatz Industrie inTsd.€ (KJE)': 'econ_turnover_industry',
    'Beschäftigte Industrie gesamt (KJE)': 'econ_count_employees_industry',
    'Produktionsindex Bau (at; 2021=100)': 'econ_prod_index_construction',
    'Technische Gesamtproduktion Bau in Tsd. € (KJE)': 'econ_total_production_construction',
    'Umsatzindex Bau (2021=100)': 'econ_turnover_index_construction',
    'Umsatz Bau in Tsd. € (KJE)': 'econ_turnover_construction',
    'Beschäftigte Bau gesamt (KJE)': 'econ_count_employees_construction',
    'Umsatzindex - Großhandel (G46; nom.; 2021=100)': 'econ_turnover_index_wholesale',
    'Umsatzindex - Einzelhandel (G47; nom.; 2021=100)': 'econ_turnover_index_retail',
    'Beschäftigtenindex Handel  (2021=100)': 'econ_count_employees_retail',
    'Umsatzindex - KfZ-Handel (G45; nom.; 2021=100)': 'econ_turnover_index_car_retail',
    'Einfuhren Insgesamt in €': 'econ_imports_total_EUR',
    'Einfuhren Insg. SITC-3 Brennstoffe, Energie in €': 'econ_imports_energy_EUR'
}

# Setup functions (reused from previous pipelines - consistent)
def setup_pandas_options():
    """Configure pandas display options for better data inspection."""
    pd.set_option('display.max_columns', None)
    pd.set_option('display.max_rows', 100)  
    pd.set_option('display.width', None)
    warnings.filterwarnings('ignore')

def setup_project_paths(data_subfolder):
    """Set up project directory paths for any data source."""
    PROJECT_ROOT = Path.cwd().parent if 'notebooks' in str(Path.cwd()) else Path.cwd()
    RAW_DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / data_subfolder
    PROCESSED_DATA_PATH = PROJECT_ROOT / 'data' / 'processed'
    PROCESSED_DATA_PATH.mkdir(parents=True, exist_ok=True)
    return PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH

def print_notebook_header(notebook_name, objective, raw_path, processed_path):
    """Print standardized notebook start information."""
    print("="*60)
    print(f"NOTEBOOK: {notebook_name}")
    print("="*60)
    print(f"Start Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Raw Data Path: {raw_path}")
    print(f"Processed Data Path: {processed_path}")
    print(f"Objective: {objective}")
    print("="*60)

# Setup execution
setup_pandas_options()
PROJECT_ROOT, RAW_DATA_PATH, PROCESSED_DATA_PATH = setup_project_paths(DATA_SUBFOLDER)
print_notebook_header(NOTEBOOK_NAME, OBJECTIVE, RAW_DATA_PATH, PROCESSED_DATA_PATH)

print("Cell 1: Configuration and Setup - READY")

NOTEBOOK: 06: Economic Indicators Data Ingestion
Start Time: 2025-10-03 05:59:33
Raw Data Path: c:\Users\paulr\Desktop\Capstone\data\raw\economy
Processed Data Path: c:\Users\paulr\Desktop\Capstone\data\processed
Objective: Process Statistik Austria economic indicators (monthly)
18 economic variables including production indices, trade, employment
Date range: 2009-2025

IMPORTANT ARCHITECTURAL DECISION:
Economic data is MONTHLY ONLY - no daily or weekly rows will be created.
Consistent with climate, production, and gas data architecture.
Economic variables can only be used in monthly-level analyses.

DATE FORMAT: mmm.yy (e.g., Jän.09) with Austrian German month abbreviations
COMPLEX STRUCTURE: Multiple header rows and footer metadata require careful skipping.
Cell 1: Configuration and Setup - READY


In [13]:
# =============================================================================
# CELL 2: Missing Values Standardization Function (with Quality Control)
# =============================================================================

def standardize_missing_values(df, additional_missing=None, show_quality_control=True):
    """
    Convert various missing value representations to pandas NaN.
    
    Args:
        df (DataFrame): Input dataframe
        additional_missing (list): Extra missing indicators beyond defaults
        show_quality_control (bool): Whether to display quality control output
    
    Returns:
        DataFrame: Dataframe with standardized missing values
        dict: Report of found missing patterns
    """
    missing_indicators = [
        'N/A', 'n/a', 'NA', 'na',
        '', '-', '--', '---',
        'NULL', 'null', 'Null',
        'NaN', 'nan', '#N/A'
    ]
    
    if additional_missing:
        missing_indicators.extend(additional_missing)
    
    if show_quality_control:
        print("\nMISSING VALUES STANDARDIZATION - QUALITY CONTROL:")
        print("-" * 55)
        print("Missing indicators standardized:")
        print("  ", ', '.join([f"'{ind}'" for ind in missing_indicators]))
        print()
    
    found_patterns = {}
    df_clean = df.copy()
    
    for col in df_clean.columns:
        original_nulls = df_clean[col].isnull().sum()
        df_clean[col] = df_clean[col].replace(missing_indicators, np.nan)
        new_nulls = df_clean[col].isnull().sum()
        converted_count = new_nulls - original_nulls
        
        if converted_count > 0:
            found_patterns[col] = {
                'original_nulls': original_nulls,
                'converted_missing': converted_count,
                'total_nulls': new_nulls
            }
    
    if show_quality_control:
        if found_patterns:
            print("CONVERSION RESULTS BY COLUMN:")
            for col, pattern in found_patterns.items():
                print(f"  {col}:")
                print(f"    Original nulls: {pattern['original_nulls']}")
                print(f"    Converted missing: {pattern['converted_missing']}")
                print(f"    Total nulls: {pattern['total_nulls']}")
        else:
            print("No missing value patterns found for conversion")
        print("-" * 55)
    
    return df_clean, found_patterns

print("Cell 2: Missing Values Standardization Function - READY")

Cell 2: Missing Values Standardization Function - READY


In [14]:
# =============================================================================
# CELL 3: Economic Indicators File Discovery
# =============================================================================

def discover_economy_file(data_path, filename):
    """
    Discover single economic indicators data file.
    
    Args:
        data_path (Path): Path to raw data directory
        filename (str): Expected filename
    
    Returns:
        dict: File information dictionary
    """
    file_path = data_path / filename
    
    if file_path.exists():
        return {
            'filename': filename,
            'path': str(file_path),
            'exists': True
        }
    else:
        return {
            'filename': filename,
            'path': str(file_path),
            'exists': False
        }

def display_file_discovery_results(file_info):
    """Display results of economic indicators file discovery."""
    print("Economic indicators file discovery:")
    print(f"  File: {file_info['filename']}")
    print(f"  Exists: {file_info['exists']}")
    print(f"  Path: {file_info['path']}")
    
    if file_info['exists']:
        print("Ready to proceed with economic data loading")
    else:
        print("ERROR: Economic indicators file not found!")

# Execute file discovery
economy_file_info = discover_economy_file(RAW_DATA_PATH, ECON_FILE_NAME)
display_file_discovery_results(economy_file_info)

print("Cell 3: Economic Indicators File Discovery - COMPLETE")

Economic indicators file discovery:
  File: table_2025-09-16_13-30-40.xlsx
  Exists: True
  Path: c:\Users\paulr\Desktop\Capstone\data\raw\economy\table_2025-09-16_13-30-40.xlsx
Ready to proceed with economic data loading
Cell 3: Economic Indicators File Discovery - COMPLETE


In [15]:
# =============================================================================
# CELL 4: Load and Clean Economic Indicators Data (REVISED v2)
# =============================================================================

def is_valid_date_format(value):
    """
    Check if value matches Austrian economic date format mmm.yy
    
    Args:
        value: String to check
    
    Returns:
        bool: True if valid date format, False otherwise
    """
    if pd.isna(value) or value == '':
        return False
    try:
        s = str(value).strip()
        # Valid format: 3-4 chars + '.' + 2 chars (e.g., "Jän.09" or "Sept.25")
        if '.' not in s:
            return False
        parts = s.split('.')
        return len(parts) == 2 and len(parts[0]) <= 4 and len(parts[1]) == 2
    except:
        return False

def parse_austrian_economic_date(date_str):
    """
    Parse Austrian date format mmm.yy to datetime.
    Example: 'Jän.09' → 2009-01-01
    
    Assumes all dates are 2000s (09 → 2009, 25 → 2025)
    
    Args:
        date_str (str): Date string in format "Jän.09"
    
    Returns:
        datetime: Parsed datetime object (first day of month)
    """
    if pd.isna(date_str):
        return pd.NaT
    
    try:
        month_abbr, year = str(date_str).strip().split('.')
        month_num = GERMAN_MONTHS.get(month_abbr, month_abbr)
        return pd.to_datetime(f"20{year}-{month_num}-01")
    except Exception as e:
        # Silently return NaT for invalid dates (footer text)
        return pd.NaT

def load_economy_data(file_path, skiprows, column_mapping):
    """
    Load economic indicators data from Excel and select relevant columns.
    Automatically detects end of data by validating date format.
    
    Args:
        file_path (str/Path): Path to economic indicators Excel file
        skiprows (list): Row indices to skip (metadata rows)
        column_mapping (dict): Mapping of source to target column names
    
    Returns:
        tuple: (dataframe, metadata)
    """
    try:
        # Load Excel file without nrows - will filter later
        df = pd.read_excel(file_path, skiprows=skiprows)
        
        print("RAW EXCEL LOAD:")
        print(f"  Initial shape: {df.shape}")
        print()
        
        # Drop first column (empty index column)
        df = df.iloc[:, 1:]
        
        print(f"After dropping column 0: {df.shape}")
        print()
        
        # Strip whitespace from column names
        df.columns = df.columns.str.strip()
        
        # ===================================================================
        # VALIDATION CHECKPOINT 1: Column Detection
        # ===================================================================
        print("COLUMN VALIDATION:")
        print("-" * 60)
        date_col_name = df.columns[0]
        print(f"  First column name: '{date_col_name}'")
        
        # ===================================================================
        # Filter to valid data rows (only rows with valid date format)
        # ===================================================================
        print("\nDATA ROW FILTERING:")
        print("-" * 60)
        first_col = df.columns[0]
        
        # Find valid rows (rows with valid date format mmm.yy)
        valid_rows = df[first_col].apply(is_valid_date_format)
        rows_before = len(df)
        df = df[valid_rows].reset_index(drop=True)
        rows_after = len(df)
        
        print(f"  Rows before filtering: {rows_before}")
        print(f"  Rows after filtering: {rows_after}")
        print(f"  Removed {rows_before - rows_after} footer/invalid rows")
        print()
        
        # ===================================================================
        # VALIDATION CHECKPOINT 2: Data Range
        # ===================================================================
        print("DATA RANGE VALIDATION:")
        print("-" * 60)
        print(f"  First date value (raw): {df.iloc[0, 0]}")
        print(f"  Last date value (raw): {df.iloc[-1, 0]}")
        print(f"  Shape after filtering: {df.shape}")
        print(f"  Expected: (~200, 19)")
        print()
        
        # ===================================================================
        # Parse Austrian dates
        # ===================================================================
        print("DATE PARSING:")
        print("-" * 60)
        
        df[first_col] = df[first_col].apply(parse_austrian_economic_date)
        
        # ===================================================================
        # VALIDATION CHECKPOINT 3: Date Parsing Success
        # ===================================================================
        print(f"  First parsed date: {df[first_col].iloc[0]}")
        print(f"  Last parsed date: {df[first_col].iloc[-1]}")
        print(f"  Date range: {df[first_col].min()} to {df[first_col].max()}")
        failed_parses = df[first_col].isna().sum()
        print(f"  Failed parses: {failed_parses}")
        if failed_parses > 0:
            print(f"  ⚠️ WARNING: {failed_parses} dates could not be parsed!")
        else:
            print(f"  ✓ All dates parsed successfully")
        print()
        
        # ===================================================================
        # VALIDATION CHECKPOINT 4: Column Matching
        # ===================================================================
        print("COLUMN SELECTION VALIDATION:")
        print("-" * 60)
        
        # Adjust column mapping to use actual first column name
        adjusted_mapping = column_mapping.copy()
        if first_col != 'Berichtszeitraum':
            adjusted_mapping[first_col] = adjusted_mapping.pop('Berichtszeitraum')
        
        actual_cols = set(df.columns)
        expected_cols = set(adjusted_mapping.keys())
        missing_cols = expected_cols - actual_cols
        extra_cols = actual_cols - expected_cols
        
        if missing_cols:
            print(f"  ⚠️ Missing columns: {missing_cols}")
        if extra_cols:
            print(f"  ⚠️ Unexpected columns: {extra_cols}")
        if not missing_cols and not extra_cols:
            print(f"  ✓ All {len(expected_cols)} expected columns present")
        print()
        
        # Clean missing values BEFORE renaming
        df_clean, missing_patterns = standardize_missing_values(df, show_quality_control=True)
        
        # Rename columns to standardized names
        df_clean.rename(columns=adjusted_mapping, inplace=True)
        
        print(f"\nCOLUMN RENAMING:")
        print(f"  Renamed {len(adjusted_mapping)} columns with econ_ prefix")
        print()
        
        # Create metadata
        metadata = {
            'filename': Path(file_path).name,
            'rows': len(df_clean),
            'columns': list(df_clean.columns),
            'missing_patterns_found': missing_patterns,
            'date_range': (df_clean['date'].min(), df_clean['date'].max()) if len(df_clean) > 0 else (None, None)
        }
        
        print(f"DATA SUMMARY:")
        print(f"  Final shape: {df_clean.shape}")
        print(f"  Date range: {metadata['date_range'][0].strftime('%Y-%m-%d')} to {metadata['date_range'][1].strftime('%Y-%m-%d')}")
        print()
        
        print("SAMPLE DATA:")
        print(df_clean.head(10))
        
        return df_clean, metadata
        
    except Exception as e:
        print(f"ERROR loading economic data: {e}")
        import traceback
        traceback.print_exc()
        return None, {'error': str(e)}

# Execute data loading
if economy_file_info['exists']:
    economy_df, economy_metadata = load_economy_data(
        economy_file_info['path'],
        SKIPROWS,
        COLUMN_MAPPING
    )
    
    if economy_df is None:
        print(f"ERROR loading economic data: {economy_metadata['error']}")
else:
    print("Skipping data load - file not found")

print("Cell 4: Load and Clean Economic Indicators Data - COMPLETE")

RAW EXCEL LOAD:
  Initial shape: (227, 20)

After dropping column 0: (227, 19)

COLUMN VALIDATION:
------------------------------------------------------------
  First column name: 'Unnamed: 1'

DATA ROW FILTERING:
------------------------------------------------------------
  Rows before filtering: 227
  Rows after filtering: 200
  Removed 27 footer/invalid rows

DATA RANGE VALIDATION:
------------------------------------------------------------
  First date value (raw): Jän.09
  Last date value (raw): Aug.25
  Shape after filtering: (200, 19)
  Expected: (~200, 19)

DATE PARSING:
------------------------------------------------------------
  First parsed date: 2009-01-01 00:00:00
  Last parsed date: 2025-08-01 00:00:00
  Date range: 2009-01-01 00:00:00 to 2025-08-01 00:00:00
  Failed parses: 0
  ✓ All dates parsed successfully

COLUMN SELECTION VALIDATION:
------------------------------------------------------------
  ✓ All 19 expected columns present


MISSING VALUES STANDARDIZATION

In [16]:
economy_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 19 columns):
 #   Column                                Non-Null Count  Dtype         
---  ------                                --------------  -----         
 0   date                                  200 non-null    datetime64[ns]
 1   econ_prod_index_industry              55 non-null     float64       
 2   econ_consumer_price_index             115 non-null    float64       
 3   econ_exports_total_EUR                198 non-null    float64       
 4   econ_turnover_index_commercial_sales  54 non-null     float64       
 5   econ_count_overnight_stays            199 non-null    float64       
 6   econ_turnover_industry                5 non-null      float64       
 7   econ_count_employees_industry         5 non-null      float64       
 8   econ_prod_index_construction          55 non-null     float64       
 9   econ_total_production_construction    5 non-null      float64       
 10  ec